In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
#fiz seeds so runs are responsible 
torch.manual_seed(0)
np.random.seed(0)

In [ ]:
data = np.load(r'C:\Users\kteja\Downloads\learning-physical-dynamics-across-resolutions-group-12\learning-physical-dynamics-across-resolutions-group-12\trainv')
X = data['field'][:, None, :].astype(np.float32)   # (240, 1, 128)
Y = data['target'][:, None, :].astype(np.float32)  # (240, 1, 128)

In [ ]:
print("X:", X.shape, "Y:", Y.shape)

In [ ]:
# grid-locked CNN: kernel is in grid points, so its effective
# receptive field shrinks as the grid is refined
class GridLockedCNN(nn.Module):
    def __init__(self, hidden=64, kernel_size=5, num_layers=4):
        super().__init__()
        layers = [nn.Conv1d(1, hidden, kernel_size,
                            padding=kernel_size // 2,
                            padding_mode='circular'),
                  nn.ReLU()]
        for _ in range(num_layers - 1):
            layers += [nn.Conv1d(hidden, hidden, kernel_size,
                                 padding=kernel_size // 2,
                                 padding_mode='circular'),
                       nn.ReLU()]
        layers.append(nn.Conv1d(hidden, 1, kernel_size,
                                padding=kernel_size // 2,
                                padding_mode='circular'))
        self.net = nn.Sequential(*layers)
# residual: predict the update, not the full state
    def forward(self, u):
        return u + self.net(u)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)

In [ ]:
model = GridLockedCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.5)

In [ ]:
X_t = torch.tensor(X)
Y_t = torch.tensor(Y)
loader = DataLoader(TensorDataset(X_t, Y_t), batch_size=32, shuffle=True)

In [ ]:
def relative_l2(pred, target):
    num = torch.norm(pred - target, dim=(1, 2))
    den = torch.norm(target, dim=(1, 2)) + 1e-8
    return (num / den).mean()

In [ ]:
## identity gives a floor: any real model has to beat this
with torch.no_grad():
    identity_err = relative_l2(X_t, Y_t).item()
print(f"Identity baseline (predict u_t as u_t+1): {identity_err:.6f}")

In [ ]:
num_epochs = 300
for epoch in range(num_epochs):
    model.train()
    total = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb)
        loss = relative_l2(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * len(xb)
    scheduler.step()
    total /= len(X_t)

    if epoch % 20 == 0 or epoch == num_epochs - 1:
        print(f"Epoch {epoch:3d}: train rel-L2 = {total:.6f}")

In [ ]:
torch.save(model.state_dict(), 'cnn_128.pt')
print("Saved to cnn_128.pt")